# Testing Notebook

# Enformer

In [ ]:
import torch
from enformer_pytorch import Enformer, seq_indices_to_one_hot

model = Enformer.from_hparams(
    dim=1536,
    depth=11,
    heads=8,
    output_heads=dict(human=5313, mouse=1643),  # noqa: C408
    target_length=896,
)

seq = torch.randint(0, 5, (1, 196_608))
one_hot = seq_indices_to_one_hot(seq)

output, embeddings = model(one_hot, return_embeddings=True)

embeddings  # (1, 896, 3072)

tensor([[[ 0.2573,  0.1184,  0.3862,  ...,  0.2613, -0.0000,  0.0315],
         [ 0.0012,  0.4613,  0.4624,  ..., -0.0172,  0.0690,  0.1773],
         [-0.0000,  0.0455,  0.3596,  ...,  0.1237, -0.0104,  0.0446],
         ...,
         [-0.1636,  0.1695,  0.0000,  ...,  0.0493,  0.1665, -0.0124],
         [ 0.0872,  0.1877,  0.5801,  ...,  0.3235, -0.0951, -0.0811],
         [ 0.1893, -0.0970,  0.0843,  ...,  0.2897,  0.3593,  0.2840]]],
       grad_fn=<MulBackward0>)

In [7]:
print(one_hot.shape)

torch.Size([1, 196608, 4])


In [ ]:
# In [1]: Imports and setup
import random
import torch

from embpy.models.dna_models import EnformerWrapper

# 1) Initialize the Enformer wrapper on CPU
wrapper = EnformerWrapper("EleutherAI/enformer-official-rough")
wrapper.load(device=torch.device("cpu"))
print("Enformer model loaded. Sequence length =", wrapper.SEQUENCE_LENGTH)


# 2) Helper to generate a random DNA string of exactly SEQUENCE_LENGTH
def random_dna_fixed(length: int) -> str:  # noqa: D103
    return "".join(random.choice("ACGTN") for _ in range(length))


# 3) Build a small batch of random sequences
batch_size = 4
seq_len = wrapper.SEQUENCE_LENGTH
dna_batch = [random_dna_fixed(seq_len) for _ in range(batch_size)]
print("Batch sizes:", [len(s) for s in dna_batch])

# 4) Run embed_batch with mean pooling
embeddings = wrapper.embed_batch(dna_batch, pooling_strategy="mean")

print("\nembed_batch outputs:")
for i, emb in enumerate(embeddings):
    print(f"  Sequence {i}: shape {emb.shape}, first 5 dims {emb[:5]}")

Enformer model loaded. Sequence length = 196608
Batch sizes: [196608, 196608, 196608, 196608]

embed_batch outputs:
  Sequence 0: shape (3072,), first 5 dims [-0.09349184 -0.00098717 -0.00434084 -0.09515201  0.21379483]
  Sequence 1: shape (3072,), first 5 dims [-0.12184776 -0.00174527  0.0444397  -0.06221178  0.24697961]
  Sequence 2: shape (3072,), first 5 dims [-0.11219615 -0.00108458  0.07503723 -0.03661817  0.17436083]
  Sequence 3: shape (3072,), first 5 dims [-0.07893984 -0.00125954  0.10306348 -0.10213654  0.22425897]


# Borzoi Embeddings

In [ ]:
from embpy.models.dna_models import BorzoiWrapper
import torch


def random_dna(seq: int):  # noqa: D103
    return "".join(random.choice("ACTG") for _ in range(seq))


seq = random_dna(200_000)

wrapper = BorzoiWrapper("johahi/borzoi-replicate-0")
wrapper.load(device=torch.device("cpu"))

embs = wrapper.embed(input=seq, pooling_strategy="mean")

print("Shape of embeddings :", embs.shape)
print("Value of embeddings (after pooling) :", embs)

Running Borzoi model …
tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]])
torch.Size([1, 1536, 6144])
Shape of embeddings : (1536,)
Value of embeddings (after pooling) : [-0.48280635 -0.20134836 -0.583343   ...  0.16279209 -1.2780379
 -0.4207624 ]


In [ ]:
# In [1]: imports
import torch
import torch.nn.functional as F

from embpy.models.dna_models import BorzoiWrapper
from borzoi_pytorch import Borzoi


# 1) Utility to generate a fixed‐length DNA string
def random_dna_fixed(length: int) -> str:  # noqa: D103
    return "".join(random.choice("ACGT") for _ in range(length))


# 2) Instantiate & load your wrapper
wrapper = BorzoiWrapper("johahi/borzoi-replicate-0")
wrapper.load(device=torch.device("cpu"))

# 3) Generate a small batch of length-524288 sequences
batch_size = 3
seq_length = 524_288
dna_batch = [random_dna_fixed(seq_length) for _ in range(batch_size)]
print("All sequences length:", [len(s) for s in dna_batch])

# 4) Embed with your wrapper’s batch API
embs_wrapper = wrapper.embed_batch(dna_batch, pooling_strategy="mean")
print("\nWrapper.embed_batch results:")
for i, emb in enumerate(embs_wrapper):
    print(f"  Seq {i}: shape {emb.shape}")
    print(emb)

# 5) Cross-check via the raw Borzoi API (no pad/crop needed since L_in == SEQUENCE_LENGTH)
model = Borzoi.from_pretrained("johahi/borzoi-replicate-0").eval()


def direct_embed(seq: str) -> torch.Tensor:  # noqa: D103
    # map to indices & one-hot
    mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
    idx = torch.tensor([[mapping[b] for b in seq]], dtype=torch.long)  # (1,524288)
    oh = F.one_hot(idx, num_classes=4).permute(0, 2, 1).float()  # (1,4,524288)

    # directly run get_embs_after_crop (no pad/crop needed)
    with torch.no_grad():
        embs = model.get_embs_after_crop(oh)  # (1, hidden_dim, bins)

    # mean‐pool over bins
    trunk = embs.squeeze(0)  # (hidden_dim, bins)
    return trunk.mean(dim=1)  # → (hidden_dim,)


print("\nDirect API results:")
for i, seq in enumerate(dna_batch):
    vec = direct_embed(seq).cpu().numpy()
    print(f"  Seq {i}: shape {vec.shape}")
    print(vec)

All sequences length: [524288, 524288, 524288]

Wrapper.embed_batch results:
  Seq 0: shape (1536,)
[-0.44751954 -0.12284581 -1.280599   ...  0.23352616 -1.2240677
 -0.5466354 ]
  Seq 1: shape (1536,)
[-0.43250263 -0.23620397 -1.2487166  ...  0.2902791  -1.2746342
 -0.4591093 ]
  Seq 2: shape (1536,)
[-0.44117403 -0.14102112 -1.1964587  ...  0.34102002 -1.2130623
 -0.5561335 ]

Direct API results:
  Seq 0: shape (1536,)
[-0.44751954 -0.12284581 -1.280599   ...  0.23352616 -1.2240677
 -0.5466354 ]
  Seq 1: shape (1536,)
[-0.43250263 -0.23620397 -1.2487166  ...  0.2902791  -1.2746342
 -0.4591093 ]
  Seq 2: shape (1536,)
[-0.44117403 -0.14102112 -1.1964587  ...  0.34102002 -1.2130623
 -0.5561335 ]


# Request Data from ENSEMBL to test the bed files and the FASTA sequences

In [3]:
from pyensembl import EnsemblRelease

# Load Ensembl release 109 (for GRCh38 / human)
data = EnsemblRelease(109)

# Download and index (only needed once)
data.download()
data.index()

# Fetch genes with name "TP53"
genes = data.genes_by_name("TP53")

# Access the first result
if genes:
    gene = genes[0]
    print("Gene:", gene.gene_name)
    print("Ensembl ID:", gene.gene_id)
    print("Chromosome:", gene.contig)
    print("Start:", gene.start)
    print("End:", gene.end)
    print("Strand:", gene.strand)
else:
    print("Gene not found.")

INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /Users/grpinto/Library/Caches/pyensembl/GRCh38/ensembl109/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Gene: TP53
Ensembl ID: ENSG00000141510
Chromosome: 17
Start: 7661779
End: 7687538
Strand: -


In [31]:
import requests

gene = "BRCA1"
r = requests.get(
    f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?expand=1", headers={"Content-Type": "application/json"}
)
info = r.json()

chrom = info["seq_region_name"]
start = info["start"] - 1  # BED is 0-based
end = info["end"]
strand = "+" if info["strand"] == 1 else "-"
name = info["display_name"]

bed_line = f"{chrom}\t{start}\t{end}\t{name}\t{strand}\t{name}"
print(bed_line)

17	43044294	43170245	BRCA1	-	BRCA1


In [32]:
import requests

url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?expand=1"
headers = {"Content-Type": "application/json"}

response = requests.get(url, headers=headers)

if response.ok:
    data = response.json()
    print("Gene name:", data.get("display_name"))
    print("Description:", data.get("description"))
else:
    print(f"Failed to fetch info: {response.status_code}")

Gene name: BRCA1
Description: BRCA1 DNA repair associated [Source:HGNC Symbol;Acc:HGNC:1100]


In [29]:
# Construct API URL for sequence
fasta_url = f"https://rest.ensembl.org/sequence/region/human/{chrom}:{start + 1}..{end}:{strand}"

# Request FASTA
headers = {"Content-Type": "text/x-fasta"}
response = requests.get(fasta_url, headers=headers)

# Check response and print
if response.ok:
    print(response.text)
else:
    print(f"Failed to fetch FASTA: {response.status_code}")

>chromosome:GRCh38:17:43044295:43170245:-1
...
AAACACTTCCA



# How to get information from a molecule ?

In [36]:
import requests

# Define the compound name
compound_name = "aspirin"

# Fetch compound CID
cid_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/cids/JSON"
cid_response = requests.get(cid_url)
cid = cid_response.json()["IdentifierList"]["CID"][0]

# Fetch compound properties
properties = "MolecularFormula,MolecularWeight,CanonicalSMILES,InChIKey,IUPACName"
property_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/property/{properties}/JSON"
property_response = requests.get(property_url)
property_data = property_response.json()["PropertyTable"]["Properties"][0]

# Display the information
print(f"Compound: {compound_name}")
print(f"CID: {cid}")
print(f"IUPAC Name: {property_data['IUPACName']}")
print(f"Molecular Formula: {property_data['MolecularFormula']}")
print(f"Molecular Weight: {property_data['MolecularWeight']}")
print(f"SMILES: {property_data['CanonicalSMILES']}")
print(f"InChIKey: {property_data['InChIKey']}")

Compound: aspirin
CID: 2244
IUPAC Name: 2-acetyloxybenzoic acid
Molecular Formula: C9H8O4
Molecular Weight: 180.16
SMILES: CC(=O)OC1=CC=CC=C1C(=O)O
InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N


In [38]:
import requests

molecule_name = "aspirin"
base_url = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"

# Fetch compound CID
cid_url = f"{base_url}/compound/name/{molecule_name}/cids/JSON"
cid_response = requests.get(cid_url)
if cid_response.ok:
    cid = cid_response.json()["IdentifierList"]["CID"][0]
    print(f"CID: {cid}")
else:
    print("Compound not found.")
    exit()

# Fetch compound properties
properties = "MolecularFormula,MolecularWeight,CanonicalSMILES,InChI,InChIKey,IUPACName"
prop_url = f"{base_url}/compound/cid/{cid}/property/{properties}/JSON"
prop_response = requests.get(prop_url)
if prop_response.ok:
    props = prop_response.json()["PropertyTable"]["Properties"][0]
    print("\nCompound Properties:")
    for key, value in props.items():
        print(f"{key}: {value}")
else:
    print("Failed to retrieve compound properties.")

# Fetch compound synonyms
synonyms_url = f"{base_url}/compound/cid/{cid}/synonyms/JSON"
syn_response = requests.get(synonyms_url)
if syn_response.ok:
    synonyms = syn_response.json()["InformationList"]["Information"][0]["Synonym"]
    print("\nSynonyms:")
    for synonym in synonyms[:10]:  # Display first 10 synonyms
        print(f"- {synonym}")
else:
    print("Failed to retrieve synonyms.")

CID: 2244

Compound Properties:
CID: 2244
MolecularFormula: C9H8O4
MolecularWeight: 180.16
CanonicalSMILES: CC(=O)OC1=CC=CC=C1C(=O)O
InChI: InChI=1S/C9H8O4/c1-6(10)13-8-5-3-2-4-7(8)9(11)12/h2-5H,1H3,(H,11,12)
InChIKey: BSYNRYMUTXBXSQ-UHFFFAOYSA-N
IUPACName: 2-acetyloxybenzoic acid

Synonyms:
- aspirin
- ACETYLSALICYLIC ACID
- 50-78-2
- 2-(Acetyloxy)benzoic acid
- Acetosal
- Acenterine
- Acetophen
- Acetosalin
- Aspirdrops
- Salcetogen


In [33]:
# import torch
# from evo2 import Evo2

# evo2_model = Evo2('evo2_7b')

# sequence = 'ACGT'
# input_ids = torch.tensor(
#     evo2_model.tokenizer.tokenize(sequence),
#     dtype=torch.int,
# ).unsqueeze(0).to('cuda:0')

# layer_name = 'blocks.28.mlp.l3'

# outputs, embeddings = evo2_model(input_ids, return_embeddings=True, layer_names=[layer_name])

# print('Embeddings shape: ', embeddings[layer_name].shape)

# ChemBERTa

In [9]:
import torch
from transformers import AutoTokenizer, AutoModel

# 2a) Specify the model name on Hugging Face Hub
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

# 2b) Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2c) Load the (masked‐LM / embedding) model
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()  # we’re doing inference only

RobertaModel(
  (embeddings): RobertaEmbeddings(
    (word_embeddings): Embedding(767, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): RobertaEncoder(
    (layer): ModuleList(
      (0-5): 6 x RobertaLayer(
        (attention): RobertaAttention(
          (self): RobertaSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): RobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropou

In [10]:
# Example SMILES strings
smiles_list = ["CCO", "c1ccccc1"]

# 3a) Tokenize with padding/truncation to a uniform length
#     ChemBERTa’s tokenizer will split into character‐level tokens,
#     add special [CLS]/[SEP], etc.
encodings = tokenizer(smiles_list, padding=True, truncation=True, return_tensors="pt")
# encodings is a dict: { "input_ids": Tensor(batch_size×seq_len),
#                        "attention_mask": Tensor(batch_size×seq_len) }

In [11]:
# 4a) Move to GPU if available (optional)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
encodings = {k: v.to(device) for k, v in encodings.items()}

# 4b) Forward pass (no gradient needed)
with torch.no_grad():
    outputs = model(**encodings)
    # AutoModel returns a BaseModelOutput with:
    #   last_hidden_state shape = (batch_size, seq_len, hidden_dim)
    #   (there may be other fields—e.g. pooler_output—depending on the config)
    last_hidden = outputs.last_hidden_state

print("last_hidden_state.shape =", last_hidden.shape)

last_hidden_state.shape = torch.Size([2, 6, 768])


In [12]:
# Suppose cls_embedding = hidden at index 0 for each sequence
cls_embeddings = last_hidden[:, 0, :]  # shape: (batch_size, 768)

# Or mean‐pool across the seq_len dimension (ignoring padding via attention_mask)
attention_mask = encodings["attention_mask"].unsqueeze(-1)  # (batch_size, seq_len, 1)
masked_hidden = last_hidden * attention_mask  # zero‐out padded positions
sum_hidden = masked_hidden.sum(dim=1)  # sum over seq_len → (batch_size, 768)
lengths = attention_mask.sum(dim=1)  # how many real tokens (batch_size, 1)
mean_embeddings = sum_hidden / lengths  # (batch_size, 768)

print("CLS‐based embedding:", cls_embeddings.shape)
print("Mean‐pooled embedding:", mean_embeddings.shape)

CLS‐based embedding: torch.Size([2, 768])
Mean‐pooled embedding: torch.Size([2, 768])


In [13]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

smiles = ["Cn1c(=O)c2c(ncn2C)n(C)c1=O", "CC(=O)Oc1ccccc1C(=O)O"]
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

configuration_molformer.py:   0%|          | 0.00/7.60k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- configuration_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_molformer.py:   0%|          | 0.00/39.3k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- modeling_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/ibm/MoLFormer-XL-both-10pct:
- tokenization_molformer_fast.py
- tokenization_molformer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


vocab.json:   0%|          | 0.00/41.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tensor([[-0.4407,  0.3902,  0.7989,  ..., -0.6081, -0.0200,  0.0103],
        [ 0.5943,  0.4527,  0.3437,  ...,  0.1520, -0.3464,  0.5590]])

# Testing Out Embeddings

In [ ]:
import torch
import numpy as np

# 2) Import your BioEmbedder (which under the hood will use EnformerWrapper)
from embpy.embedder import BioEmbedder


# 1) Generate a random DNA string (you can choose any length; Enformer will pad or truncate to 196 608 bp)
def random_dna(length: int) -> str:  # noqa: D103
    return "".join(random.choices(["A", "C", "G", "T"], k=length))


# For example, pick length = 200 000 (slightly above 196 608 so Enformer will center‐crop):
dna_seq = random_dna(200_000)
print("Generated DNA length:", len(dna_seq))  # → 200000


# 3) Instantiate BioEmbedder and load the Enformer model onto CPU (or "cuda" if you have a GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder = BioEmbedder(device=device)

# 4) Now call embed_gene with a dummy “gene identifier”—but since we already have the raw DNA,
#    we’ll bypass gene‐resolution and call the EnformerWrapper directly.
#    (Alternatively, if your GeneResolver can fetch a DNA string for an actual gene, you can use embed_gene.)

#   Here we manually get the Enformer wrapper from the embedder’s registry, then call embed(…).
model_name = "enformer_human_rough"
wrapper = embedder._get_model(model_name)  # this will load under the hood if not already loaded

# 5) Embed the random DNA sequence using Enformer
#    (EnformerWrapper will pad/truncate internally to exactly 196608, one-hot encode, run the model, pool)
embedding = wrapper.embed(input=dna_seq, pooling_strategy="mean")

print("Enformer embedding shape:", embedding.shape)
# Should be (3072,) because EnformerWrapper.TRUNK_OUTPUT_DIM = 3072

# 6) (Optional) Convert to float32 NumPy if needed:
embedding = embedding.astype(np.float32)

Generated DNA length: 200000


Enformer embedding shape: (3072,)


In [ ]:
import torch
import numpy as np
from embpy.embedder import BioEmbedder


# 1) Generate a random DNA string (you can choose any length; Enformer will pad or truncate to 196 608 bp)
def random_dna(length: int) -> str:  # noqa: D103
    return "".join(random.choices(["A", "C", "G", "T"], k=length))


# For example, pick length = 200 000 (slightly above 196 608 so Enformer will center‐crop):
dna_seq = random_dna(196_608)
print("Generated DNA length:", len(dna_seq))  # → 200000

# 2) Import your BioEmbedder (which under the hood will use EnformerWrapper)

# 3) Instantiate BioEmbedder and load the Enformer model onto CPU (or "cuda" if you have a GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embedder = BioEmbedder(device=device)

# 4) Now call embed_gene with a dummy “gene identifier”—but since we already have the raw DNA,
#    we’ll bypass gene‐resolution and call the EnformerWrapper directly.
#    (Alternatively, if your GeneResolver can fetch a DNA string for an actual gene, you can use embed_gene.)

#   Here we manually get the Enformer wrapper from the embedder’s registry, then call embed(…).
model_name = "borzoi_v0"
wrapper = embedder._get_model(model_name)  # this will load under the hood if not already loaded

# 5) Embed the random DNA sequence using Enformer
#    (EnformerWrapper will pad/truncate internally to exactly 196608, one-hot encode, run the model, pool)
embedding = wrapper.embed(input=dna_seq, pooling_strategy="mean")

print("Borzoi embedding shape:", embedding.shape)
# Should be (3072,) because EnformerWrapper.TRUNK_OUTPUT_DIM = 3072

# 6) (Optional) Convert to float32 NumPy if needed:
embedding = embedding.astype(np.float32)

Generated DNA length: 196608
Running Borzoi model …
torch.Size([1, 1536, 16352])
Borzoi embedding shape: (1536,)


# ESM 2 embeddings 

In [12]:
# In [1]:
from embpy.embedder import BioEmbedder
import torch

# 1) Initialize your embedder on CPU (or "cuda"):
embedder = BioEmbedder(device=torch.device("cpu"))

# 2) See which models you have available:
print("Available models:", embedder.list_available_models())

# 3) Pick an ESM2 model key from that list:
model_key = "esm2_650M"

# 4) Generate a random protein (amino‐acid) sequence:
amino_acids = "ACDEFGHIKLMNPQRSTVWY"
seq_length = 200
random_protein = "".join(random.choices(amino_acids, k=seq_length))
print(f"Random protein ({seq_length} aa):", random_protein[:50] + "…")

# 5) Load the wrapper (this will download & cache the HF weights under-the-hood):
wrapper = embedder._get_model(model_key)

# 6) Embed your sequence (default is “mean” pooling; you can also use “max” or “cls”):
embedding = wrapper.embed(random_protein, pooling_strategy="mean")

# 7) Inspect the result:
print("Embedding shape:", embedding.shape)  # e.g. (768,) or however large your ESM-2 hidden dim is

Available models: ['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']
Random protein (200 aa): NRMFPLCTWCYHGIVRYCSFTVPHPNCPKNDETRLTANEPEHPRQFWIGA…


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t33_650M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Embedding shape: (1280,)


# Chemberta

In [ ]:
import torch
import random
from transformers import AutoTokenizer, AutoModel

# 0) make everything reproducible
seed = 42
random.seed(seed)
torch.manual_seed(seed)

# 1) Pick a ChemBERTa checkpoint
model_name = "seyonec/ChemBERTa-zinc-base-v1"

# 2) Load tokenizer & model (and switch to eval)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).eval()  # <— IMPORTANT!


# 3) Tokenize your SMILES
def random_smiles(n: int) -> str:  # noqa: D103
    tokens = ["C", "N", "O"]
    return "".join(random.choices(tokens, k=n))


smiles = random_smiles(128)

inputs = tokenizer(smiles, padding=True, truncation=True, return_tensors="pt")

# 4) Forward pass (no gradient needed, dropout off because of eval())
with torch.no_grad():
    outputs = model(**inputs)

# 5) Extract the CLS embedding
cls_direct = outputs.last_hidden_state[:, 0, :].cpu()

# 6) Now do it through your wrapper
from embpy.embedder import BioEmbedder  # noqa: E402

embedder = BioEmbedder(device="cpu")
vec_cls = embedder.embed_molecule(smiles, model="chemberta_zinc_v1", pooling_strategy="cls")

# they should now be identical (up to floating‐point rounding)
print("Direct CLS:", cls_direct.shape, cls_direct[0, :5])
print("Wrapper CLS:", vec_cls.shape, vec_cls[:5])
# print("Max abs diff:", (cls_direct[0].numpy() - vec_cls).abs().max())

Direct CLS: torch.Size([1, 768]) tensor([ 0.0313, -0.8472, -0.9893,  0.2864, -0.7245])
Wrapper CLS: (768,) [ 0.03130991 -0.8472023  -0.98933     0.28640974 -0.72450495]


# MolFormer

In [1]:
from embpy.embedder import BioEmbedder
import torch

embedder = BioEmbedder(device=torch.device("cpu"))
# should list both chemberta_zinc_v1 and molformer_xl
print(embedder.list_available_models())

smiles = "CC(=O)Oc1ccccc1C(=O)O"
emb = embedder.embed_molecule(smiles, model="molformer_base", pooling_strategy="cls")
print("MolFormer embedding shape:", emb.shape)

['bert_base_uncased', 'borzoi_v0', 'borzoi_v1', 'chemberta_zinc_v1', 'enformer_human_rough', 'esm2_150M', 'esm2_35M', 'esm2_3B', 'esm2_650M', 'esm2_8M', 'minilm_l6_v2', 'molformer_base']


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


MolFormer embedding shape: (768,)


In [4]:
print(emb)

[ 5.94280064e-01  4.52650040e-01  3.43747914e-01  6.67825580e-01
 -6.99579656e-01  2.02832788e-01 -5.44053353e-02  8.19928646e-01
  2.11379856e-01 -1.15139507e-01 -6.97321951e-01  7.15679109e-01
  5.20055711e-01  2.29307845e-01 -4.24501389e-01  1.97534814e-01
  3.03925332e-02 -2.96359032e-01 -4.01238978e-01 -5.29726088e-01
 -7.38125205e-01  6.60815015e-02  3.13309371e-01  7.46078014e-01
  2.41676614e-01  4.75266218e-01 -1.43960690e+00 -2.91483134e-01
 -6.32730305e-01  7.32680500e-01 -6.61421657e-01 -6.26281261e-01
 -1.05253361e-01  3.25632334e-01  6.07707977e-01  4.24847901e-01
  3.78585994e-01 -3.01597148e-01 -6.80134296e-01 -1.79451853e-01
 -1.41719967e-01  4.31221336e-01 -1.47480192e-02 -1.45637870e-01
 -1.72145277e-01 -2.95462102e-01  1.42974734e-01 -7.01401532e-02
  2.54375458e-01  4.04036850e-01  6.48585916e-01  1.57276704e-03
  5.96743226e-01  7.34008610e-01  1.47559628e-01 -1.99199796e-01
 -3.74050140e-01  2.49511153e-01 -1.34057164e-01 -5.61859123e-02
  3.31139356e-01  4.65937

In [3]:
import torch
from transformers import AutoModel, AutoTokenizer

model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

smiles = "CC(=O)Oc1ccccc1C(=O)O"
inputs = tokenizer(smiles, padding=True, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
outputs.pooler_output

tensor([[ 5.9428e-01,  4.5265e-01,  3.4375e-01,  6.6783e-01, -6.9958e-01,
          2.0283e-01, -5.4405e-02,  8.1993e-01,  2.1138e-01, -1.1514e-01,
         -6.9732e-01,  7.1568e-01,  5.2006e-01,  2.2931e-01, -4.2450e-01,
          1.9753e-01,  3.0393e-02, -2.9636e-01, -4.0124e-01, -5.2973e-01,
         -7.3813e-01,  6.6082e-02,  3.1331e-01,  7.4608e-01,  2.4168e-01,
          4.7527e-01, -1.4396e+00, -2.9148e-01, -6.3273e-01,  7.3268e-01,
         -6.6142e-01, -6.2628e-01, -1.0525e-01,  3.2563e-01,  6.0771e-01,
          4.2485e-01,  3.7859e-01, -3.0160e-01, -6.8013e-01, -1.7945e-01,
         -1.4172e-01,  4.3122e-01, -1.4748e-02, -1.4564e-01, -1.7215e-01,
         -2.9546e-01,  1.4297e-01, -7.0140e-02,  2.5438e-01,  4.0404e-01,
          6.4859e-01,  1.5728e-03,  5.9674e-01,  7.3401e-01,  1.4756e-01,
         -1.9920e-01, -3.7405e-01,  2.4951e-01, -1.3406e-01, -5.6186e-02,
          3.3114e-01,  4.6594e-02,  5.4272e-01, -9.9487e-02,  4.9468e-01,
         -2.7091e-02,  2.3985e-01,  1.

# Checking for this API wrappers, to see if they work 

In [1]:
from embpy.resources.gene_resolver import GeneResolver

# Initialize the resolver
resolver = GeneResolver()

In [3]:
resolver = GeneResolver()
seq = resolver.get_dna_sequence("BRCA1", id_type="symbol")
print(seq)

...


In [3]:
import requests

gene = "BRCA1"

# Step 1: Get gene coordinates and metadata
lookup_url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?expand=1"
headers = {"Content-Type": "application/json"}
r = requests.get(lookup_url, headers=headers)
r.raise_for_status()
info = r.json()

# Extract coordinates
chrom = info["seq_region_name"]
start = info["start"]
end = info["end"]
strand = info["strand"]
gene_id = info["id"]
name = info["display_name"]

# BED format (0-based start)
bed_line = f"{chrom}\t{start - 1}\t{end}\t{name}\t0\t{'+' if strand == 1 else '-'}"
print("BED line:")
print(bed_line)

# Step 2: Get the DNA sequence using Ensembl's sequence endpoint
sequence_url = f"https://rest.ensembl.org/sequence/id/{gene_id}?type=genomic"
seq_response = requests.get(sequence_url, headers={"Content-Type": "text/x-fasta"})
seq_response.raise_for_status()
fasta = seq_response.text

print("\nFASTA sequence:")
print(fasta)

BED line:
17	43044294	43170245	BRCA1	0	-

FASTA sequence:
>ENSG00000012048.26 chromosome:GRCh38:17:43044295:43170245:-1
...
AAACACTTCCA



# get_protein_sequence

In [6]:
from embpy.resources.gene_resolver import GeneResolver  # Replace with actual import
import time

# Initialize
resolver = GeneResolver()

# Example identifiers to test
genes_to_test = [
    ("TP53", "symbol"),
    ("ENSG00000139618", "ensembl_id"),  # BRCA2
    ("P38398", "uniprot_id"),  # BRCA1
]

# Benchmark get_protein_sequence
print("Testing get_protein_sequence:")
for identifier, id_type in genes_to_test:
    print(f"\nFetching protein for {id_type.upper()} '{identifier}':")
    start = time.time()
    seq = resolver.get_protein_sequence(identifier, id_type)
    end = time.time()
    if seq:
        print(f"✓ Length: {len(seq)} aa | Time: {end - start:.2f}s")
        print(f"Start: {seq}")
    else:
        print("✗ Sequence not found.")

ERROR:root:Error fetching protein sequence for TP53: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=gene_exact:TP53%20AND%20organism:%22human%22&format=json&fields=accession


Testing get_protein_sequence:

Fetching protein for SYMBOL 'TP53':
✗ Sequence not found.

Fetching protein for ENSEMBL_ID 'ENSG00000139618':


ERROR:root:Error fetching protein sequence for ENSG00000139618: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=dbxref:Ensembl:ENSG00000139618&format=json&fields=accession


✗ Sequence not found.

Fetching protein for UNIPROT_ID 'P38398':
✓ Length: 1863 aa | Time: 0.23s
Start: MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKAACEFSETDVTNTEHHQPSNNDLNTTEKRAAERHPEKYQGSSVSNLHVEPCGTNTHASSLQHENSSLLLTKDRMNVEKAEFCNKSKQPGLARSQHNRWAGSKETCNDRRTPSTEKKVDLNADPLCERKEWNKQKLPCSENPRDTEDVPWITLNSSIQKVNEWFSRSDELLGSDDSHDGESESNAKVADVLDVLNEVDEYSGSSEKIDLLASDPHEALICKSERVHSKSVESNIEDKIFGKTYRKKASLPNLSHVTENLIIGAFVTEPQIIQERPLTNKLKRKRRPTSGLHPEDFIKKADLAVQKTPEMINQGTNQTEQNGQVMNITNSGHENKTKGDSIQNEKNPNPIESLEKESAFKTKAEPISSSISNMELELNIHNSKAPKKNRLRRKSSTRHIHALELVVSRNLSPPNCTELQIDSCSSSEEIKKKKYNQMPVRHSRNLQLMEGKEPATGAKKSNKPNEQTSKRHDSDTFPELKLTNAPGSFTKCSNTSELKEFVNPSLPREEKEEKLETVKVSNNAEDPKDLMLSGERVLQTERSVESSSISLVPGTDYGTQESISLLEVSTLGKAKTEPNKCVSQCAAFENPKGLIHGCSKDNRNDTEGFKYPLGHEVNHSRETSIEMEESELDAQYLQNTFKVSKRQSFAPFSNPGNAEEECATFSAHSGSLKKQS

In [3]:
from embpy.resources.gene_resolver import GeneResolver
import time

resolver = GeneResolver()

genes_to_test = [
    ("TP53", "symbol"),
    ("P04637", "uniprot_id"),
    ("ENSP00000419060", "ensembl_id"),  # <-- transcript ID, not gene ID!
]

print("Testing get_protein_sequence:\n")
for identifier, id_type in genes_to_test:
    print(f"Fetching protein for {id_type.upper()} '{identifier}':")
    start = time.time()
    sequence = resolver.get_protein_sequence(identifier=identifier, id_type=id_type)
    elapsed = time.time() - start

    if sequence:
        print(f"✓ Success | Length: {len(sequence)} aa | Time: {elapsed:.2f}s")
        print(f"First 60 aa: {sequence[:60]}...\n")
    else:
        print("✗ Sequence not found.\n")

Testing get_protein_sequence:

Fetching protein for SYMBOL 'TP53':
✓ Success | Length: 393 aa | Time: 0.32s
First 60 aa: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

Fetching protein for UNIPROT_ID 'P04637':
✓ Success | Length: 393 aa | Time: 0.12s
First 60 aa: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

Fetching protein for ENSEMBL_ID 'ENSP00000419060':


ERROR:root:Error fetching protein sequence for ENSP00000419060: 400 Client Error: Bad Request for url: https://rest.uniprot.org/uniprotkb/search?query=xref%3AEnsembl%3AENSP00000419060%20AND%20reviewed%3Atrue&format=json


✗ Sequence not found.



In [1]:
from embpy.resources.gene_resolver import GeneResolver

resolver = GeneResolver()

# Example: Get UniProt ID from Ensembl ID
ensembl_id = "ENSG00000141510"  # TP53
uniprot_id = resolver.get_uniprot_id_from_gene(ensembl_id, id_type="ensembl_id")

if uniprot_id:
    print(f"Resolved UniProt ID: {uniprot_id}")
    protein_seq = resolver.get_protein_sequence(uniprot_id, id_type="uniprot_id")
    print(f"Protein sequence length: {len(protein_seq)}\nFirst 60 aa: {protein_seq[:60]}...")
else:
    print("Could not resolve UniProt ID.")

Resolved UniProt ID: P04637
Protein sequence length: 393
First 60 aa: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...


In [1]:
from embpy.resources.gene_resolver import GeneResolver
import time

# Initialize resolver
resolver = GeneResolver()

# Example test cases: symbol, Ensembl gene ID, and UniProt accession
genes_to_test = [
    ("TP53", "symbol"),  # Human tumor suppressor gene
    ("ENSG00000141510", "ensembl_id"),  # Ensembl gene ID for TP53
    ("P04637", "uniprot_id"),  # UniProt accession for TP53 protein
]

# Run benchmark
print("🧬 Testing get_protein_sequence:\n")
for identifier, id_type in genes_to_test:
    print(f"🔍 Fetching protein for {id_type.upper()} '{identifier}':")
    start = time.time()
    sequence = resolver.get_protein_sequence(identifier=identifier, id_type=id_type)
    elapsed = time.time() - start

    if sequence:
        print(f"✅ Success | Length: {len(sequence)} aa | Time: {elapsed:.2f}s")
        print(f"🔢 Start of sequence: {sequence[:60]}...\n")
    else:
        print("❌ Failed to retrieve protein sequence.\n")

🧬 Testing get_protein_sequence:

🔍 Fetching protein for SYMBOL 'TP53':
✅ Success | Length: 393 aa | Time: 0.98s
🔢 Start of sequence: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

🔍 Fetching protein for ENSEMBL_ID 'ENSG00000141510':
✅ Success | Length: 393 aa | Time: 1.00s
🔢 Start of sequence: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...

🔍 Fetching protein for UNIPROT_ID 'P04637':
✅ Success | Length: 393 aa | Time: 0.11s
🔢 Start of sequence: MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP...



In [2]:
from embpy.resources.gene_resolver import GeneResolver  # Update to actual import if needed

# Initialize the resolver
resolver = GeneResolver()

# Example gene identifiers
genes_to_test = [
    ("TP53", "symbol"),
    ("ENSG00000141510", "ensembl_id"),  # TP53 Ensembl ID
    ("P04637", "uniprot_id"),  # TP53 UniProt ID
]

# Format string (optional – can include more keys if MyGene.info provides them)
format_str = "Gene: {symbol}. Name: {name}. Summary: {summary}"

print("🧬 Testing get_gene_description:\n")
for identifier, id_type in genes_to_test:
    print(f"🔍 {id_type.upper()} '{identifier}':")
    description = resolver.get_gene_description(identifier, id_type=id_type, format_string=format_str)
    if description:
        print(f"✅ Description:\n{description}\n")
    else:
        print("❌ Description not found.\n")

🧬 Testing get_gene_description:

🔍 SYMBOL 'TP53':
✅ Description:
Gene: TP53. Name: tumor protein p53. Summary: This gene encodes a tumor suppressor protein containing transcriptional activation, DNA binding, and oligomerization domains. The encoded protein responds to diverse cellular stresses to regulate expression of target genes, thereby inducing cell cycle arrest, apoptosis, senescence, DNA repair, or changes in metabolism. Mutations in this gene are associated with a variety of human cancers, including hereditary cancers such as Li-Fraumeni syndrome. Alternative splicing of this gene and the use of alternate promoters result in multiple transcript variants and isoforms. Additional isoforms have also been shown to result from the use of alternate translation initiation codons from identical transcript variants (PMIDs: 12032546, 20937277). [provided by RefSeq, Dec 2016].

🔍 ENSEMBL_ID 'ENSG00000141510':
✅ Description:
Gene: TP53. Name: tumor protein p53. Summary: This gene encodes a